Spline fits code

In [ ]:
# --- Plotting ---
def exp_model(t, C, A, k):
    return C + A * np.exp(-k * t)

# find the index where kappa==0
i0 = int(np.where(np.isclose(kappa_vals, 0.0))[0][0])
y_vals = []
t = np.asarray(times, float)
from scipy.interpolate import UnivariateSpline
import matplotlib.pyplot as plt

metrics = []
spline_fits = []  # Store spline fits for later plotting

# Create figure with subplots
n_plots = len(D_all)
# fig, axes = plt.subplots(n_plots, 1, figsize=(12, 4*n_plots))
# if n_plots == 1:
#     axes = [axes]

y0_common = D_all[0][0]
for idx, D in enumerate(D_all):
    y = np.asarray(D, float)
    spline = UnivariateSpline(t, y, s=len(t)*0.01)
    y_fit = spline(t)
    
    # Store for main plot
    spline_fits.append((t, y_fit))
    
    # Calculate derivative
    dy = spline.derivative()(t)
    
    # Calculate metrics
    mean_slope = np.mean(dy)
    decay_fraction = np.sum(dy < 0) / len(dy)
    oscillation = np.std(y - spline(t))
    net_change = (y[0] - y[-1]) / y[0] if y[0] != 0 else 0
    
    metrics.append({
        'kappa': kappa_vals[idx],
        'mean_slope': mean_slope,
        'decay_fraction': decay_fraction,
        'oscillation': oscillation,
        'net_change': net_change
    })

# Summary table
import pandas as pd
df = pd.DataFrame(metrics)

# Main plot with fits overlaid
plt.figure(figsize=(14, 12))
for i, kappa in enumerate(kappa_vals):
    # Get the spline fit for this curve
    t_dense, y_fit = spline_fits[i]
    
    # Plot original data as solid line
    line, = plt.plot(times, D_all[i], linewidth=3)
    ratio = kappa/(1/np.sqrt(dim_N1))
    # print(f"kappa = {kappa}")
    # print(f"\sigma^2 = {(1/np.sqrt(dim_N1))}")
    line, = plt.plot(times, D_all[i], label=fr'$g/\sigma = {ratio:.3f}$', linewidth=3)
    
    # Plot spline fit as dotted line with same color
    # plt.plot(t_dense, y_fit, '--', color=line.get_color(), linewidth=2, alpha=0.8)

plt.xlabel(r'Time', fontsize=32)
plt.ylabel('Trace distance', fontsize=32)
plt.legend(fontsize=20)
plt.tick_params(axis='both', which='major', labelsize=26)
plt.grid(True)
plt.tight_layout()
plt.show()